# Code Converter - Python to Rust

Convert Python code to optimized Rust code and run it.

In [ ]:
# imports
import os
import io
import sys
import json
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import gradio as gr

In [ ]:
# load env + client
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set. Check your environment variables and try again")

openai = OpenAI()
OPENAI_MODEL = "gpt-5-nano"

In [ ]:
# compile + run commands for Rust (release = optimized)
RUST_PROJECT_DIR = "rust_out"
compile_command = ["cargo", "build", "--release", "--manifest-path", f"{RUST_PROJECT_DIR}/Cargo.toml"]
run_command = [f"./{RUST_PROJECT_DIR}/target/release/converted"]

In [ ]:
# prompts for conversion
system_prompt = """
Your task is to convert Python code into high-performance, optimized Rust code.
Respond only with Rust code. Do not provide any explanation other than occasional comments.
Use idiomatic Rust: prefer iterators, avoid unnecessary clones, use appropriate types (e.g. u32/i64 for integers).
The Rust code must produce identical output and should be optimized for speed (e.g. release builds, minimal allocations).
"""

def user_prompt_for(python):
    return f"""
Port this Python code to Rust with the fastest possible implementation that produces identical output.

Your response will be written to src/main.rs in a Cargo project and then compiled with `cargo build --release` and executed.

Respond only with Rust code (no Cargo.toml; assume a standard binary crate with main in src/main.rs).
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]

def ensure_rust_project():
    """Ensure rust_out/ Cargo project exists with src/main.rs."""
    os.makedirs(f"{RUST_PROJECT_DIR}/src", exist_ok=True)
    cargo_toml = f"{RUST_PROJECT_DIR}/Cargo.toml"
    if not os.path.exists(cargo_toml):
        with open(cargo_toml, "w", encoding="utf-8") as f:
            f.write('[package]\nname = "converted"\nversion = "0.1.0"\nedition = "2021"\n')

def write_output(code):
    ensure_rust_project()
    with open(f"{RUST_PROJECT_DIR}/src/main.rs", "w", encoding="utf-8") as f:
        f.write(code)

def convert(python):
    response = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages_for(python),
        reasoning_effort="high",
    )
    reply = response.choices[0].message.content
    reply = reply.replace("```rust", "").replace("```", "")
    return reply

In [ ]:
# sample python for testing
py_sample = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output


def run_rust(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True, cwd=os.getcwd())
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True, cwd=os.getcwd())
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr or e.stdout}"

In [ ]:
# Gradio UI
with gr.Blocks(theme=gr.themes.Monochrome(), title="Port from Python to Rust") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python Original Code",
                value=py_sample,
                language="python",
                lines=30,
            )
        with gr.Column(scale=6):
            rust = gr.Code(
                label="Rust (generated)", value="", language="rust", lines=26
            )
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        port = gr.Button("Convert to Rust", elem_classes=["convert-btn"])
        rust_run = gr.Button("Run Rust", elem_classes=["run-btn", "rust"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python Result", lines=10)
        with gr.Column(scale=6):
            rust_out = gr.TextArea(label="Rust output", lines=10)

    port.click(fn=convert, inputs=[python], outputs=[rust])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    rust_run.click(fn=run_rust, inputs=[rust], outputs=[rust_out])

ui.launch(inbrowser=True)